# Descarga de metadatos de Last.fm
Ejecutar desde mineria/. Reutiliza los Parquet de descargas anteriores. Repetir si se amplía el catálogo, cambian las fechas o quedaron consultas pendientes. Las semillas y K no afectan esta descarga.

In [1]:
from pathlib import Path
import os, re, time, unicodedata
from getpass import getpass
import requests, duckdb
import pandas as pd
import numpy as np
N_ARTISTAS = 1000
FECHA_VAL = '2013-10-01'
FECHA_TEST = '2014-01-01'
CONSULTAR_API = True
MAX_CANCIONES_ARTISTA = 2
ESPERA_API = 1.1
ruta_clean = Path('../data/clean')
ruta_parquet = Path('../data/parquet')
ruta_tmp = Path('../.tmp')
ruta_tmp.mkdir(exist_ok=True)
ruta_parquet.mkdir(exist_ok=True)

archivo_eventos = ruta_clean / 'events_clean.parquet'
archivo_audio = ruta_clean / 'acoustic_clean.parquet'
archivo_artistas = ruta_parquet / 'artists.parquet'
archivo_tracks = ruta_parquet / 'tracks.parquet'
for archivo in [archivo_eventos, archivo_artistas]:
    if not archivo.exists():
        raise FileNotFoundError(f'No encontré {archivo.resolve()}')

con = duckdb.connect()
con.execute("SET memory_limit = '4GB'")
con.execute('SET threads = 4')
con.execute(f"SET temp_directory = '{ruta_tmp.resolve().as_posix()}'")
con.execute(f"CREATE OR REPLACE VIEW events_clean AS SELECT * FROM '{archivo_eventos.as_posix()}' WHERE timestamp >= epoch(TIMESTAMP '2010-01-01')")


def q(sql): return con.sql(sql).df()
def unix(fecha): return int(pd.Timestamp(fecha, tz='UTC').timestamp())

def nombre_normalizado(valor):
    valor = unicodedata.normalize('NFKD', str(valor).casefold())
    valor = ''.join(x for x in valor if not unicodedata.combining(x))
    return re.sub(r'\s+', ' ', valor).strip()

nombres_artistas = (pd.read_parquet(archivo_artistas, columns=['artist_id', 'artist_name'])
                    .dropna(subset=['artist_id']).drop_duplicates('artist_id')
                    .set_index('artist_id')['artist_name'])
print('Datos preparados. Artistas con nombre:', len(nombres_artistas))

Datos preparados. Artistas con nombre: 3190371


In [2]:
def catalogo(corte):
    return q(f"""
        SELECT artist_id FROM events_clean
        WHERE timestamp < {unix(corte)} AND artist_id IS NOT NULL AND user_id IS NOT NULL
        GROUP BY artist_id ORDER BY COUNT(DISTINCT user_id) DESC, artist_id
        LIMIT {N_ARTISTAS}
    """).artist_id
def canciones_historicas(artistas, corte):
    if not archivo_tracks.exists():
        print('No encontré tracks.parquet; omito las consultas de canciones.')
        return pd.DataFrame(columns=['artist_id', 'track_id', 'track_name', 'artist_name', 'escuchas'])
    columnas = set(q(f"DESCRIBE SELECT * FROM '{archivo_tracks.as_posix()}'").column_name)
    if not {'track_id','track_name'}.issubset(columnas):
        print('tracks.parquet no tiene track_id y track_name; omito canciones.')
        return pd.DataFrame(columns=['artist_id', 'track_id', 'track_name', 'artist_name', 'escuchas'])
    con.register('catalogo_canciones', pd.DataFrame({'artist_id': np.asarray(artistas, dtype='int64')}))
    canciones = q(f"""
        WITH conteo AS (
            SELECT e.artist_id, e.track_id, COUNT(*) AS escuchas
            FROM events_clean e JOIN catalogo_canciones c USING(artist_id)
            WHERE e.timestamp < {unix(corte)} AND e.track_id IS NOT NULL
            GROUP BY e.artist_id, e.track_id
        ), orden AS (
            SELECT *, ROW_NUMBER() OVER(
                PARTITION BY artist_id ORDER BY escuchas DESC, track_id
            ) AS puesto
            FROM conteo
        )
        SELECT o.artist_id, o.track_id, t.track_name, o.escuchas
        FROM orden o JOIN '{archivo_tracks.as_posix()}' t USING(track_id)
        WHERE o.puesto <= {MAX_CANCIONES_ARTISTA}
    """)
    con.unregister('catalogo_canciones')
    canciones['artist_name'] = canciones.artist_id.map(nombres_artistas)
    return canciones.dropna(subset=['artist_name', 'track_name'])

In [3]:
# 3.A. Carga de etiquetas de la notebook anterior (evita repetir 200 consultas).
ARCHIVO_TAGS_VIEJO = ruta_parquet / 'artist_tags_lastfm_enriquecido.parquet'
ARCHIVO_TAGS = ruta_parquet / 'artist_tags_lastfm_enriquecido.parquet'
ARCHIVO_TAGS_ESTADOS = ruta_parquet / 'artist_tags_lastfm_estados.parquet'
CACHE_INFO = ruta_parquet / 'artist_info_actual_lastfm.parquet'
CACHE_SIM = ruta_parquet / 'artist_similares_actual_lastfm.parquet'
CACHE_TOP = ruta_parquet / 'artist_toptracks_actual_lastfm.parquet'
CACHE_TRACKS = ruta_parquet / 'track_info_y_tags_actual_lastfm.parquet'
URL_LASTFM = 'https://ws.audioscrobbler.com/2.0/'
API_KEY = None

def api(metodo, **kwargs):
    """Consulta con reintentos y pausa. No ponemos la clave en el notebook."""
    global API_KEY
    if API_KEY is None:
        API_KEY = os.getenv('LASTFM_API_KEY', '').strip() or getpass('Clave API Last.fm: ').strip()
    if not API_KEY:
        raise RuntimeError('No se indicó la clave API')
    params = dict(method=metodo, api_key=API_KEY, format='json', **kwargs)
    for intento in range(4):
        try:
            r = requests.get(URL_LASTFM, params=params, timeout=25)
            if r.status_code in [429, 500, 502, 503, 504]:
                raise requests.RequestException(f'HTTP {r.status_code}')
            r.raise_for_status()
            datos = r.json()
            if 'error' in datos:
                if int(datos['error']) == 6:
                    return {}  # Artista/canción no encontrada.
                if int(datos['error']) in [8, 11, 16, 29]:
                    raise requests.RequestException(f"API {datos['error']}: {datos.get('message')}")
                raise ValueError(f"Error Last.fm {datos['error']}: {datos.get('message')}")
            time.sleep(ESPERA_API)
            return datos
        except (requests.RequestException, ValueError):
            if intento == 3:
                raise
            time.sleep(min(30, 3 * (2 ** intento)))

def lista(x):
    if isinstance(x, list): return x
    return [x] if isinstance(x, dict) else []

def numero(x):
    try: return float(x)
    except (TypeError, ValueError): return 0.0

def leer_cache(archivo, columnas):
    if archivo.exists():
        df = pd.read_parquet(archivo)
        if set(columnas).issubset(df): return df
    return pd.DataFrame(columns=columnas)

def descargar_tags(ids):
    columnas = ['artist_id', 'artist_name', 'tag', 'peso']
    tags = leer_cache(ARCHIVO_TAGS, columnas)
    estados = leer_cache(ARCHIVO_TAGS_ESTADOS, ['artist_id', 'estado'])
    conocidos = set(estados.loc[estados.estado.isin(['ok', 'sin_tags']), 'artist_id'].astype(int))
    # Si existe una tabla de etiquetas previa pero no sus estados, no repetir lo ya disponible.
    conocidos |= set(tags.artist_id.dropna().astype(int))
    pendientes = [int(i) for i in ids if int(i) not in conocidos]
    print('Etiquetas pendientes:', len(pendientes))
    if not CONSULTAR_API:
        return tags
    for i, aid in enumerate(pendientes, 1):
        nombre = nombres_artistas.get(aid)
        if not isinstance(nombre, str): continue
        try:
            datos = api('artist.getTopTags', artist=nombre)
            nuevas = [dict(artist_id=aid, artist_name=nombre,
                           tag=str(x.get('name','')).strip().casefold(), peso=numero(x.get('count')))
                      for x in lista(datos.get('toptags', {}).get('tag'))]
            nuevas = [x for x in nuevas if x['tag'] and x['peso'] > 0]
            if nuevas: tags = pd.concat([tags, pd.DataFrame(nuevas)], ignore_index=True)
            estados = pd.concat([estados, pd.DataFrame([dict(artist_id=aid,
                estado='ok' if nuevas else 'sin_tags')])], ignore_index=True)
        except Exception as e:
            print('Falló la consulta de etiquetas:', nombre, str(e)[:100])
        if i % 10 == 0 or i == len(pendientes):
            tags = tags.drop_duplicates(['artist_id', 'tag'], keep='last')
            estados = estados.drop_duplicates('artist_id', keep='last')
            tags.to_parquet(ARCHIVO_TAGS, index=False)
            estados.to_parquet(ARCHIVO_TAGS_ESTADOS, index=False)
            print(f'Etiquetas: {i}/{len(pendientes)}')
    return tags

def descargar_artistas(ids):
    info = leer_cache(CACHE_INFO, ['artist_id', 'listeners_actual', 'playcount_actual'])
    similares = leer_cache(CACHE_SIM, ['artist_id', 'similar_name', 'similitud'])
    top = leer_cache(CACHE_TOP, ['artist_id', 'track_name', 'puesto_actual'])
    hechos = set(info.artist_id.dropna().astype(int))
    pendientes = [int(i) for i in ids if int(i) not in hechos]
    print('Metadatos de artistas pendientes:', len(pendientes))
    if not CONSULTAR_API: return info, similares, top
    for i, aid in enumerate(pendientes, 1):
        nombre = nombres_artistas.get(aid)
        if not isinstance(nombre, str): continue
        # Solo marcamos como completo un artista si se procesaron las tres consultas.
        try:
            d_info = api('artist.getInfo', artist=nombre, autocorrect=1).get('artist', {})
            d_sim = api('artist.getSimilar', artist=nombre, limit=20, autocorrect=1)
            d_top = api('artist.getTopTracks', artist=nombre, limit=10, page=1, autocorrect=1)
            stats = d_info.get('stats', {})
            info = pd.concat([info, pd.DataFrame([{
                'artist_id': aid, 'artist_name': nombre,
                'listeners_actual': numero(stats.get('listeners')),
                'playcount_actual': numero(stats.get('playcount')),
            }])], ignore_index=True)
            similares_nuevos = [{
                'artist_id': aid, 'similar_name': s.get('name', ''),
                'similitud': numero(s.get('match'))
            } for s in lista(d_sim.get('similarartists', {}).get('artist'))]
            if similares_nuevos:
                similares = pd.concat([similares, pd.DataFrame(similares_nuevos)], ignore_index=True)
            tops = [{
                'artist_id': aid, 'track_name': t.get('name', ''),
                'puesto_actual': j, 'playcount_actual': numero(t.get('playcount')),
            } for j,t in enumerate(lista(d_top.get('toptracks', {}).get('track')),1)]
            if tops: top = pd.concat([top, pd.DataFrame(tops)], ignore_index=True)
        except Exception as e:
            print('No se pudo completar:', nombre, str(e)[:120])
        if i % 10 == 0 or i == len(pendientes):
            info = info.drop_duplicates('artist_id', keep='last')
            similares = similares.drop_duplicates(['artist_id','similar_name'], keep='last')
            top = top.drop_duplicates(['artist_id','track_name'], keep='last')
            info.to_parquet(CACHE_INFO, index=False)
            similares.to_parquet(CACHE_SIM, index=False)
            top.to_parquet(CACHE_TOP, index=False)
            print(f'Metadatos de artistas: {i}/{len(pendientes)}')
    return info, similares, top

def descargar_canciones(candidatas):
    info = leer_cache(CACHE_TRACKS, ['artist_id', 'track_id', 'track_name', 'duracion_ms',
                                    'listeners_actual', 'tags_json'])
    hechos = set(pd.to_numeric(info.track_id, errors='coerce').dropna().astype(int))
    pendientes = candidatas[~candidatas.track_id.isin(hechos)]
    print('Canciones pendientes:', len(pendientes))
    if not CONSULTAR_API: return info
    filas = []
    import json
    for i, t in enumerate(pendientes.itertuples(index=False), 1):
        try:
            d = api('track.getInfo', artist=t.artist_name, track=t.track_name, autocorrect=1).get('track', {})
            e = api('track.getTopTags', artist=t.artist_name, track=t.track_name, autocorrect=1)
            tags = [{'tag': x.get('name', '').strip().casefold(),
                     'peso': numero(x.get('count'))}
                    for x in lista(e.get('toptags', {}).get('tag'))]
            filas.append({'artist_id': int(t.artist_id), 'track_id': int(t.track_id),
                          'track_name': t.track_name,
                          'duracion_ms': numero(d.get('duration')),
                          'listeners_actual': numero(d.get('listeners')),
                          'tags_json': json.dumps(tags, ensure_ascii=False)})
        except Exception as error:
            print('Canción sin respuesta:', t.track_name, str(error)[:100])
        if i % 10 == 0 or i == len(pendientes):
            if filas:
                info = pd.concat([info, pd.DataFrame(filas)], ignore_index=True)
            info = info.drop_duplicates('track_id', keep='last')
            info.to_parquet(CACHE_TRACKS, index=False)
            filas = []
            print(f'Canciones: {i}/{len(pendientes)}')
    return info

In [4]:
catalogos = {fecha: catalogo(fecha) for fecha in (FECHA_VAL, FECHA_TEST)}
ids = sorted(set().union(*catalogos.values()))
canciones = pd.concat([canciones_historicas(ids_corte, fecha)
                       for fecha, ids_corte in catalogos.items()]).drop_duplicates('track_id')
tags = descargar_tags(ids)
info, similares, top = descargar_artistas(ids)
tracks = descargar_canciones(canciones)
print('Catálogo combinado:', len(ids), '| Con etiquetas:', tags.artist_id.isin(ids).groupby(tags.artist_id).any().sum())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

No encontré tracks.parquet; omito las consultas de canciones.
No encontré tracks.parquet; omito las consultas de canciones.
Etiquetas pendientes: 84
Etiquetas: 10/84
Etiquetas: 20/84
Etiquetas: 30/84
Etiquetas: 40/84
Etiquetas: 50/84
Etiquetas: 60/84
Etiquetas: 70/84
Etiquetas: 80/84
Etiquetas: 84/84
Metadatos de artistas pendientes: 814
Metadatos de artistas: 10/814
Metadatos de artistas: 20/814
Metadatos de artistas: 30/814
Metadatos de artistas: 40/814
Metadatos de artistas: 50/814
Metadatos de artistas: 60/814
Metadatos de artistas: 70/814
Metadatos de artistas: 80/814
Metadatos de artistas: 90/814
Metadatos de artistas: 100/814
Metadatos de artistas: 110/814
Metadatos de artistas: 120/814
Metadatos de artistas: 130/814
Metadatos de artistas: 140/814
Metadatos de artistas: 150/814
Metadatos de artistas: 160/814
Metadatos de artistas: 170/814
Metadatos de artistas: 180/814
Metadatos de artistas: 190/814
Metadatos de artistas: 200/814
Metadatos de artistas: 210/814
Metadatos de artis